# Description

This notebook implements general functions for feature engineering, combined with few manual features. It implements a simple LGBM model that is trained upon, where the top 100 features is selected, and then a new LGBM model is trained using these features only.

Running this code locally yields 0.8964 (Private) and 0.86401 (Public), which places this model in Top1 for both Private and Public. The performance is slightly reduced whilst running this on Kaggle Notebook, however the Private value still places it in Top1.

# Feature Engineering

First we load the relevant packages, training and test dataset provided in the competition. Then the dataset is analysed checking missing, outliers, and correlation between variables.

Then 2 general functions are created that 

1. Makes basic transformations of the variables
2. Combines variables

In [ ]:
## Load packages
library(dplyr)
library(lightgbm)

## Load Training and Test data
TrainingData <- read.csv("/kaggle/input/GiveMeSomeCredit/cs-training.csv")
TestData <- read.csv("/kaggle/input/GiveMeSomeCredit/cs-test.csv")

## Remove coloumn X
TrainingData <- TrainingData %>% select(-X)

## Outlier
TrainingData$age[which(TrainingData$age == 0)] <- round(mean(TrainingData$age[TrainingData$age != 0]))

## Manual Variables
TrainingData <- TrainingData %>% mutate(TotalPastDue = NumberOfTime30.59DaysPastDueNotWorse + 
                                          NumberOfTime60.89DaysPastDueNotWorse + NumberOfTimes90DaysLate,
                                        DebtRatio1000= ifelse(MonthlyIncome < 1000, NA, DebtRatio))

## Function for general transformations
feature_engineering <- function(data, variables, operations = c("subtract")) {
  # data: The input data frame
  # variables: A vector of variable names for feature engineering
  # operations: A vector of operations to perform ("subtract", "divide", "log", "square", ...)
  
  for (var in variables) {
    if (var %in% colnames(data)) {
      for (operation in operations) {
        if (operation == "subtract") {
          new_var <- paste0(var, "_minus_mean")
          data[[new_var]] <- data[[var]] - mean(data[[var]], na.rm = TRUE)
        } else if (operation == "divide" && mean(data[[var]], na.rm = TRUE) != 0) {
          new_var <- paste0(var, "_divided_by_mean")
          data[[new_var]] <- data[[var]] / mean(data[[var]], na.rm = TRUE)
        } else if (operation == "log") {
          new_var <- paste0("log_", var)
          data[[new_var]] <- ifelse(data[[var]] == 0, NA, log(data[[var]]))
        } else if (operation == "square") {
          new_var <- paste0(var, "_squared")
          data[[new_var]] <- data[[var]]^2
        } else {
          warning("Unsupported operation '", operation, "'. Skipping.")
        }
      }
    } else {
      warning("Variable '", var, "' not found in the data.")
    }
  }
  
  return(data)
}

## Function for combining 2 variables
arithmetic_operations <- function(data, num_var1, num_var2, operations = c("divide")) {
  # data: The input data frame
  # num_var1, num_var2: Names of the numeric variables for arithmetic operation
  # operations: A vector of arithmetic operations to perform ("multiply", "subtract", "divide", "add")
  
  new_vars <- character(length(operations))
  
  if (num_var1 %in% colnames(data) && num_var2 %in% colnames(data)) {
    for (i in seq_along(operations)) {
      operation <- operations[i]
      
      new_vars[i] <- switch(
        operation,
        multiply = paste0(num_var1, "_times_", num_var2),
        subtract = paste0(num_var1, "_minus_", num_var2),
        divide = paste0(num_var1, "_divided_by_", num_var2),
        add = paste0(num_var1, "_plus_", num_var2),
        warning("Unsupported operation '", operation, "'.")
      )
      
      if (operation == "multiply") {
        data[[new_vars[i]]] <- data[[num_var1]] * data[[num_var2]]
      } else if (operation == "subtract") {
        data[[new_vars[i]]] <- data[[num_var1]] - data[[num_var2]]
      } else if (operation == "divide") {
        data[[new_vars[i]]] <- ifelse(is.na(data[[num_var2]]) | data[[num_var2]] == 0, NA, data[[num_var1]] / data[[num_var2]])
      } else if (operation == "add") {
        data[[new_vars[i]]] <- data[[num_var1]] + data[[num_var2]]
      } else {
        warning("Error during arithmetic operation.")
      }
    }
  } else {
    warning("One or both of the variables not found in the data.")
  }
  
  return(cbind(data))
}

## Select variables to transform
variables_to_engineer <- colnames(TrainingData %>% select(-c(SeriousDlqin2yrs)))

## Transform variables using subtract, divide, log and square.
TrainingData <- feature_engineering(TrainingData, variables_to_engineer, operation = c("subtract","divide","log","square"))

## Generate which variables to combine (We dont use the newly generated variables from previous step)
variable_combinations <- combn(variables_to_engineer, 2, simplify = TRUE)

## Loop over all combinations
for (i in seq_len(ncol(variable_combinations))) {
  var1 <- variable_combinations[1, i]
  var2 <- variable_combinations[2, i]
  TrainingData <- arithmetic_operations(TrainingData, var1, var2, operations = c("multiply", "subtract", "divide", "add"))
}


####### Variables for TestData
TestData <- TestData %>% mutate(TotalPastDue = NumberOfTime30.59DaysPastDueNotWorse + 
                                  NumberOfTime60.89DaysPastDueNotWorse + NumberOfTimes90DaysLate,
                                DebtRatio1000= ifelse(MonthlyIncome < 1000, NA, DebtRatio))


variables_to_engineer <- colnames(TestData %>% select(-c(X,SeriousDlqin2yrs)))

TestData <- feature_engineering(TestData, variables_to_engineer, operation = c("subtract","divide","log","square"))

variable_combinations <- combn(variables_to_engineer, 2, simplify = TRUE)

for (i in seq_len(ncol(variable_combinations))) {
  var1 <- variable_combinations[1, i]
  var2 <- variable_combinations[2, i]
  
  TestData <- arithmetic_operations(TestData, var1, var2, operations = c("multiply", "subtract", "divide", "add"))
}

# Model Part 1 (LGBM)

First we train a model using LGBM, which has been partly optimized for the TestData:

1. Leaves (12): The maximum number of leaves in one tree. Higher values inceases model complexity but runs into overfitting.
2. Learning Rate (0.013): Step size shrinkage used in each boosting iteration. A small value generally makes the optimazation more robust, but then requires more boosting rounds to reach an optimized model.
3. Feature Fraction (0.8): Proportion of features to consider in each boosting round. This is used to add some randomness into the model. Usually helps prevent overfitting.
4. Lambda L1 (0.1): Regularization term on weights. Adds a penalty term to the model for the absolute value of the coefficients. This works well when you have many variables, where most of them doesnt hold any unique performance.
5. Lambda L2 (10): Regularization term on weights. Adds a penalty term to the model for the squared values of the coefficients. This discourages large weights and helps prevent overfitting.
6. Number of boosting rounds (750): A too high number of boosting rounds will lead to overfitting, however the amount of boosting rounds also highly depend on the Learning Rate. A higher Learning Rate would typically require fewer boosting rounds.

Things to note: One should be careful using L2 regularization in this dataset, since the manually created variable TotalPastDue is highly correlated with SeriousDlqin2yrs, thus we dont want the model to not put hight weight on this variable.

In [ ]:
# Create a sparse matrix without removing NA rows as LGBM handles NA 
sparse_matrix <- model.matrix.lm(SeriousDlqin2yrs ~ ., TrainingData, na.action = "na.pass")

# Create the training data list
train <- list(data = sparse_matrix, label = as.numeric(TrainingData$SeriousDlqin2yrs))


# Set parameters for model training
train_params <- list(
  num_leaves = 12L
  , learning_rate = 0.013
  , objective = "binary"
  , feature_fraction = c(0.8)
  , use_missing = TRUE
  , metric = "AUC"
  , lambda_l1 = 0.1
  , lambda_l2 = 10
)

## LGBM model
bst <- lightgbm(
  data = train$data
  , params = train_params
  , label = train$label
  , nrounds = 750L
)


# Model Part 2 (LGBM)

Using the above trained LGBM model we extract the top 100 features based on importance. Using these 100 features a new LGBM model is now trained with optimized parameters:

1. Leaves (12)
2. Learning Rate (0.0135)
3. Feature Fraction (0.7)
4. Lambda L1 (0.1)
5. Lambda L2 (4) 
6. Number of boosting rounds (1000)
7. Bagging Frequency (10): Every 10th iteration only 0.8 (Bagging Fraction) * 100% of the dataset is used for the next 10 iterations.
8. Bagging Fraction (0.8)

Thins to note: Bagging gave this model the last predictive power to go from Top2 to Top1. The Lambda L2 has been reduced to allow a higher weight on the star variable TotalPastDue.

In [ ]:

## Feature importance
feature_importance <- lgb.importance(model = bst)

## Select top 100 features from LGBM model
selected_columns <- c("SeriousDlqin2yrs", feature_importance$Feature[1:100])

## Create new Training and Test data using only these features
TrainingData <- TrainingData[, selected_columns, drop = FALSE]
TestData <- TestData[, c("X",selected_columns), drop = FALSE]

# Create a sparse matrix without removing NA rows
sparse_matrix <- model.matrix.lm(SeriousDlqin2yrs ~ ., TrainingData, na.action = "na.pass")

# Create the training data list
train <- list(data = sparse_matrix, label = as.numeric(TrainingData$SeriousDlqin2yrs))

# Set parameters for model training
train_params <- list(
  num_leaves = 12L
  , learning_rate = 0.0135
  , objective = "binary"
  , feature_fraction = c(0.7)
  , use_missing = TRUE
  , metric = "AUC"
  , lambda_l1 = 0.1
  , lambda_l2 = 4
  , bagging_freq = 10
  , bagging_fraction = 0.8
)

#--------------------Basic Training using lightgbm----------------
print("Training lightgbm with sparseMatrix")
bst <- lightgbm(
  data = train$data
  , params = train_params
  , label = train$label
  , nrounds = 1000L
)

## Prepare the testdata
TestData2 <- TestData %>% select(-c(X))

# Create a sparse matrix without removing NA rows
sparse_matrix <- model.matrix.lm(SeriousDlqin2yrs ~ ., TestData2, na.action = "na.pass")

# Create the test data list
test <- list(
  data = sparse_matrix,
  label = as.numeric(TestData2$SeriousDlqin2yrs)
)

#--------------------Basic prediction using lightgbm--------------
# You can do prediction using the following line
# You can put in Matrix, sparseMatrix, or lgb.Dataset
pred <- predict(bst, test$data)


Result <- data.frame(Id = as.integer(TestData$X), Probability = pred)

write.csv(Result,"/kaggle/working/LGBM_Summit.csv", row.names = FALSE)